# Caption Evaluation Tool
Human-in-the-loop review of auto-generated fracture captions.

**Instructions:**
1. Set `INPUT_CSV` and `OUTPUT_CSV` paths below
2. Run all cells
3. For each object: review images + captions, edit if needed, then click **Confirm & Next**
4. Progress is saved after every confirmation — safe to interrupt and resume

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV  = "caption_breakingbad_train_new.csv"   # path to your raw captions CSV
OUTPUT_CSV = "caption_breakingbad_train_reviewed.csv"  # where reviewed results are saved

# How many caption variants to show per object (set None to show all)
MAX_CAPTIONS_PER_OBJECT = None

# Max image width in pixels for display
IMAGE_DISPLAY_WIDTH = 420

In [ ]:
import ast, json, os, sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image
import io, base64

def _img_to_b64(path, max_width=IMAGE_DISPLAY_WIDTH):
    """Load image, resize to max_width, return base64 data-URI."""
    try:
        img = Image.open(path).convert("RGB")
        w, h = img.size
        if w > max_width:
            img = img.resize((max_width, int(h * max_width / w)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        return None

def _parse_images(val):
    """Parse images_used column (JSON list or Python literal)."""
    if isinstance(val, list):
        return val
    val = str(val).strip()
    try:
        return json.loads(val)
    except Exception:
        pass
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

print("✔ Helpers loaded")

In [ ]:
# ── Load CSVs ──────────────────────────────────────────────────────────────
df_in = pd.read_csv(INPUT_CSV, sep=",")
df_in.columns = [c.strip() for c in df_in.columns]
print(f"Input CSV: {len(df_in)} rows, columns: {list(df_in.columns)}")

# Group by object path — each group is one object with N caption variants
groups = []
for obj_path, grp in df_in.groupby("path", sort=False):
    grp = grp.reset_index(drop=True)
    if MAX_CAPTIONS_PER_OBJECT is not None:
        grp = grp.head(MAX_CAPTIONS_PER_OBJECT)
    groups.append((obj_path, grp))
print(f"Objects to review: {len(groups)}")

# Load existing reviewed output (for resume)
if os.path.isfile(OUTPUT_CSV):
    df_reviewed = pd.read_csv(OUTPUT_CSV)
    reviewed_paths = set(df_reviewed["path"].unique())
    print(f"Resuming — {len(reviewed_paths)} objects already reviewed in '{OUTPUT_CSV}'")
else:
    df_reviewed = pd.DataFrame()
    reviewed_paths = set()
    print(f"Starting fresh — output will be saved to '{OUTPUT_CSV}'")

In [ ]:
# ── Main Evaluation Widget ─────────────────────────────────────────────────

# Filter out already-reviewed objects
pending = [(p, g) for p, g in groups if p not in reviewed_paths]
total   = len(groups)
done    = total - len(pending)

if not pending:
    print("🎉 All objects already reviewed! Check your output CSV.")
else:
    # State
    state = {"idx": 0}  # index into `pending`

    # ── Layout widgets ─────────────────────────────────────────────────────
    progress_label = widgets.HTML()
    path_label     = widgets.HTML()
    class_label    = widgets.HTML()
    status_out     = widgets.Output()   # flash messages

    images_out     = widgets.Output()   # rendered images
    captions_box   = widgets.VBox([])   # dynamic caption editors

    btn_confirm = widgets.Button(
        description="✔ Confirm & Next",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px")
    )
    btn_skip = widgets.Button(
        description="⟶ Skip",
        button_style="warning",
        layout=widgets.Layout(width="120px", height="40px")
    )

    nav_bar = widgets.HBox([btn_confirm, btn_skip])

    main_ui = widgets.VBox([
        progress_label,
        path_label,
        class_label,
        widgets.HTML("<hr style='margin:6px 0'>"),
        images_out,
        # widgets.HTML("<b style='font-size:14px'>Caption variants — edit as needed:</b>"),
        widgets.HTML("<b style='font-size:14px'>Caption pair:</b>"),
        captions_box,
        widgets.HTML("<hr style='margin:6px 0'>"),
        nav_bar,
        status_out,
    ], layout=widgets.Layout(padding="12px", border="1px solid #ccc",
                              border_radius="8px", max_width="900px"))

    # ── Render one object ──────────────────────────────────────────────────
    def render_current():
        idx = state["idx"]
        if idx >= len(pending):
            with status_out:
                clear_output()
                print("🎉 All done! Every object has been reviewed.")
            btn_confirm.disabled = True
            btn_skip.disabled    = True
            return

        obj_path, grp = pending[idx]
        n_done_total  = done + idx

        # Progress
        progress_label.value = (
            f"<b>Progress: {n_done_total} / {total} objects reviewed</b> "
            f"&nbsp;|&nbsp; <span style='color:#888'>Remaining: {total - n_done_total}</span>"
        )

        # Object info
        short_path = str(obj_path)
        if len(short_path) > 90:
            short_path = "…" + short_path[-87:]
        obj_class = str(grp["class"].iloc[0]) if "class" in grp.columns else ""
        path_label.value  = f"<span style='font-family:monospace;font-size:12px;color:#555'>{short_path}</span>"
        class_label.value = f"<b>Class:</b> {obj_class}"

        # Images — collect unique image list across all caption variants
        all_imgs = []
        seen = set()
        for _, row in grp.iterrows():
            imgs = _parse_images(row.get("images_used", "[]"))
            for p in imgs:
                if p not in seen:
                    seen.add(p)
                    all_imgs.append(p)

        with images_out:
            clear_output(wait=True)
            # 2-column × 3-row grid; each cell is fixed size so all 6 slots
            # are always visible even when fewer images are present.
            CELL_W, CELL_H = 420, 240   # px per grid cell
            grid_style = (
                "display:grid;"
                "grid-template-columns:repeat(2,1fr);"
                "grid-template-rows:repeat(3,auto);"
                "gap:8px;"
                "margin-bottom:12px;"
                f"max-width:{CELL_W*2+16}px;"
            )
            cell_style = (
                "display:flex;flex-direction:column;align-items:center;"
                "justify-content:center;"
                f"min-height:{CELL_H}px;"
                "background:#f8f8f8;"
                "border:1px solid #ddd;"
                "border-radius:6px;"
                "overflow:hidden;"
                "padding:4px;"
            )
            html_parts = [f"<div style='{grid_style}'>"]
            any_found = False
            # Always render exactly 6 cells (2×3), padding with empty slots
            for i in range(6):
                if i < len(all_imgs):
                    img_path = all_imgs[i]
                    uri = _img_to_b64(img_path, max_width=CELL_W)
                    fname = Path(img_path).name
                    if uri:
                        any_found = True
                        html_parts.append(
                            f"<div style='{cell_style}'>"
                            f"<img src='{uri}' style='max-width:100%;max-height:{CELL_H-28}px;object-fit:contain;border-radius:3px'/>"
                            f"<div style='font-size:10px;color:#888;margin-top:4px;text-align:center;word-break:break-all'>{fname}</div>"
                            f"</div>"
                        )
                    else:
                        html_parts.append(
                            f"<div style='{cell_style}'>"
                            f"<span style='color:red;font-size:11px'>⚠ Missing:<br>{fname}</span>"
                            f"</div>"
                        )
                else:
                    # Empty placeholder cell to keep grid shape
                    html_parts.append(f"<div style='{cell_style}'></div>")
            html_parts.append("</div>")
            if not any_found:
                html_parts = ["<div style='color:orange'>⚠ No images found for this object.</div>"]
            display(HTML("".join(html_parts)))

        # Caption editors — one textarea per caption variant
        editors = []
        for _, row in grp.iterrows():
            cap_idx = int(row.get("caption_idx", 0))
            cap_val = str(row.get("caption", "")) if not pd.isna(row.get("caption", "")) else ""
            parse_err = str(row.get("parse_error", "")).upper() == "TRUE"
            conf  = row.get("confidence", "")

            label_html = widgets.HTML(
                f"<b>Variant {cap_idx}</b>"
                + (f" &nbsp;<span style='color:gray'>confidence: {conf}</span>" if conf else "")
                + (" &nbsp;<span style='color:red'>[parse error]</span>" if parse_err else "")
            )
            ta = widgets.Textarea(
                value=cap_val,
                layout=widgets.Layout(width="860px", height="60px")
            )
            editors.append((cap_idx, ta, row))
            captions_box.children = captions_box.children + (widgets.VBox([label_html, ta]),)

        # Reset captions_box
        new_children = []
        for cap_idx, ta, row in editors:
            cap_val = str(row.get("caption", "")) if not pd.isna(row.get("caption", "")) else ""
            parse_err = str(row.get("parse_error", "")).upper() == "TRUE"
            conf  = row.get("confidence", "")
            label_html = widgets.HTML(
                f"<b>Variant {cap_idx}</b>"
                + (f" &nbsp;<span style='color:gray'>confidence: {conf}</span>" if conf else "")
                + (" &nbsp;<span style='color:red'>[parse error]</span>" if parse_err else "")
            )
            ta = widgets.Textarea(
                value=cap_val,
                layout=widgets.Layout(width="860px", height="60px")
            )
            new_children.append((ta, row))   # store ta reference for save
            captions_box.children = ()

        # Rebuild editors list cleanly
        state["editors"] = []
        vbox_children = []
        for i, row in grp.iterrows():
            cap_idx   = int(row.get("caption_idx", 0))
            cap_val   = str(row.get("caption", "")) if not pd.isna(row.get("caption", "")) else ""
            parse_err = str(row.get("parse_error", "")).upper() == "TRUE"
            conf      = str(row.get("confidence", ""))
            # if i==1:
            #     bs = "Truth"
            # else:
            #     bs = "Predict"
            label_html = widgets.HTML(
                f"<b>Caption {cap_idx}</b>"
                + (f" &nbsp;<span style='color:gray'>confidence: {conf}</span>" if conf else "")
                + (" &nbsp;<span style='color:red;font-size:11px'> ⚠ parse error in original</span>" if parse_err else "")
            )
            ta = widgets.Textarea(
                value=cap_val,
                layout=widgets.Layout(width="860px", height="64px")
            )
            state["editors"].append((cap_idx, ta, row))
            vbox_children.append(widgets.VBox([label_html, ta]))

        captions_box.children = tuple(vbox_children)

        with status_out:
            clear_output()

    # ── Save current object rows to CSV ────────────────────────────────────
    def save_current(skip=False):
        global df_reviewed, reviewed_paths
        idx = state["idx"]
        obj_path, grp = pending[idx]

        rows_to_save = []
        for cap_idx, ta, orig_row in state.get("editors", []):
            row_dict = orig_row.to_dict()
            if not skip:
                row_dict["caption"]          = ta.value.strip()
                row_dict["human_reviewed"]   = True
                row_dict["human_edited"]     = (ta.value.strip() != str(orig_row.get("caption", "")).strip())
            else:
                row_dict["human_reviewed"]   = False
                row_dict["human_edited"]     = False
            rows_to_save.append(row_dict)

        new_df = pd.DataFrame(rows_to_save)

        # Append to output CSV
        write_header = not os.path.isfile(OUTPUT_CSV)
        new_df.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

        reviewed_paths.add(obj_path)
        df_reviewed = pd.concat([df_reviewed, new_df], ignore_index=True)

        with status_out:
            clear_output()
            action = "Skipped" if skip else "Saved"
            print(f"✔ {action}: {Path(obj_path).name}  →  {OUTPUT_CSV}")

    # ── Button handlers ────────────────────────────────────────────────────
    def on_confirm(b):
        save_current(skip=False)
        state["idx"] += 1
        render_current()

    def on_skip(b):
        save_current(skip=True)
        state["idx"] += 1
        render_current()

    btn_confirm.on_click(on_confirm)
    btn_skip.on_click(on_skip)

    # ── Initial render ─────────────────────────────────────────────────────
    display(main_ui)
    render_current()

In [ ]:
if os.path.isfile(OUTPUT_CSV):
    df_out = pd.read_csv(OUTPUT_CSV)
    n_obj     = df_out["path"].nunique()
    n_rows    = len(df_out)
    n_reviewed = df_out.get("human_reviewed", pd.Series(dtype=bool)).sum() if "human_reviewed" in df_out.columns else "?"
    n_edited   = df_out.get("human_edited",   pd.Series(dtype=bool)).sum() if "human_edited"   in df_out.columns else "?"
    print(f"Output CSV: {OUTPUT_CSV}")
    print(f"  Objects : {n_obj}")
    print(f"  Rows    : {n_rows}")
    print(f"  human_reviewed=True : {n_reviewed}")
    print(f"  human_edited=True   : {n_edited}")
    display(df_out.tail(6))
else:
    print("No output CSV yet.")